# Namma Space: frames -> COLMAP -> Gaussian splat
Run the cells **top to bottom**. Before starting: **Runtime > Change runtime type > T4 GPU**.
Every long step saves its result to Google Drive, so if Colab disconnects you can re-run from the top and it will skip finished steps.

## 1. Setup (Drive, GPU check, settings)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, os, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)

# --- SETTINGS ---
ROOM = "room1"
ZIP_NAME = "room1_images.zip"          # frames zip you uploaded
MAX_ITERS = 15000                      # 15000 is fine for a test; use 30000 for the final run
WORK = "/content/work"
IMAGES = f"{WORK}/images"
NS = f"{WORK}/ns"
OUT = f"{WORK}/outputs"

# Find your Drive folder even with the trailing spaces in "Clg /namma /"
hits = glob.glob(f"/content/drive/MyDrive/Clg*/namma*/{ZIP_NAME}")
assert hits, "Zip not found. Check the folder and file name."
ZIP_PATH = hits[0]
DRIVE_DIR = os.path.dirname(ZIP_PATH)
BACKUP = f"{DRIVE_DIR}/{ROOM}_backup"
os.makedirs(BACKUP, exist_ok=True)
print("Zip:", ZIP_PATH); print("Backups go to:", BACKUP)

## 2. Install tools (about 5 to 10 min; ignore red pip warnings)

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y colmap ffmpeg > /dev/null
pip -q install nerfstudio
colmap -h | head -n 2
ns-train --help | head -n 3

## 3. Unzip the frames

In [ ]:
import zipfile, shutil
shutil.rmtree(IMAGES, ignore_errors=True)
os.makedirs(IMAGES, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(IMAGES)
# Handle a zip that contains a nested "images" folder
nested = os.path.join(IMAGES, "images")
if os.path.isdir(nested):
    for f in os.listdir(nested): shutil.move(os.path.join(nested, f), IMAGES)
    os.rmdir(nested)
frames = sorted(glob.glob(f"{IMAGES}/*.jp*g") + glob.glob(f"{IMAGES}/*.png"))
print(len(frames), "frames", frames[:2])

## 4. COLMAP camera poses (the slow step)
Uses **sequential matching** (made for video) on the CPU. Expect roughly 10 to 25 min. The cell is quiet for long stretches; that is normal.
If a previous run was saved to Drive, it is restored instead.

In [ ]:
import time
if os.path.exists(f"{BACKUP}/ns/transforms.json") and not os.path.exists(f"{NS}/transforms.json"):
    print("Restoring COLMAP result from Drive...")
    shutil.copytree(f"{BACKUP}/ns", NS)
if not os.path.exists(f"{NS}/transforms.json"):
    t=time.time()
    !ns-process-data images --data {IMAGES} --output-dir {NS} --matching-method sequential --no-gpu --verbose
    print(f"COLMAP took {(time.time()-t)/60:.1f} min")

## 5. Check how many frames got poses (want about 90% or more)

In [ ]:
import json
tf = json.load(open(f"{NS}/transforms.json"))
n_ok, n_all = len(tf["frames"]), len(frames)
print(f"{n_ok} of {n_all} frames registered ({100*n_ok/n_all:.0f}%)")
if n_ok/n_all >= 0.9: print("GOOD: go ahead and train.")
elif n_ok/n_all >= 0.5: print("OK-ish: it will train but may have holes.")
else: print("BAD: re-film slower with more overlap, or extract frames at a higher rate.")
shutil.rmtree(f"{BACKUP}/ns", ignore_errors=True)
shutil.copytree(NS, f"{BACKUP}/ns")
print("COLMAP result backed up to Drive.")

## 6. Train the Gaussian splat (20 to 40 min on a T4)

In [ ]:
t=time.time()
!ns-train splatfacto --data {NS} --output-dir {OUT} --experiment-name {ROOM} \
  --max-num-iterations {MAX_ITERS} --vis tensorboard --viewer.quit-on-train-completion True
print(f"Training took {(time.time()-t)/60:.1f} min")

## 7. Export the splat (.ply)

In [ ]:
cfgs = sorted(glob.glob(f"{OUT}/**/config.yml", recursive=True))
assert cfgs, "No trained model found. Re-run step 6."
CFG = cfgs[-1]; print("Using", CFG)
EXPORT = f"{WORK}/export"
!ns-export gaussian-splat --load-config {CFG} --output-dir {EXPORT}
plys = glob.glob(f"{EXPORT}/*.ply"); print(plys)
for p in plys: print(p, round(os.path.getsize(p)/1e6,1), "MB")

## 8. Save to Drive

In [ ]:
for p in glob.glob(f"{EXPORT}/*.ply"):
    shutil.copy(p, f"{DRIVE_DIR}/{ROOM}.ply")
shutil.copy(CFG, f"{BACKUP}/config.yml")
print("Saved:", f"{DRIVE_DIR}/{ROOM}.ply")
print("Next: download it, test it in the viewer, and compress it if it is large.")